Código 304: Análisis Cronológico y Estacionalidad

Este bloque de código realiza la conversión temporal, agrupa la siniestralidad por períodos y visualiza la evolución de las métricas de éxito.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ===========================================================================
# 304.1 CARGA, SANITIZACIÓN Y NORMALIZACIÓN TEMPORAL
# ===========================================================================
# Es imperativo cargar el dataset. Ajustar la ruta según tu infraestructura local.
path_raw = r'../raw_data/nyswcb_claims.csv'
df = pd.read_csv(path_raw, low_memory=False)

# Saneamiento de cabeceras (procedimiento estándar del Capítulo 1)
df.columns = [c.strip().replace('\r', '').replace('\n', '') for c in df.columns]

# Conversión coercitiva al tipo datetime para el Accident Date
df['Accident Date'] = pd.to_datetime(df['Accident Date'], errors='coerce')

# Creamos variables derivadas para el análisis de estacionalidad y tendencia
df['Year'] = df['Accident Date'].dt.year
df['Month'] = df['Accident Date'].dt.month

# Filtramos la ventana temporal de madurez digital (2010-2024) para evitar sesgos
df_ts = df[(df['Year'] >= 2010) & (df['Year'] <= 2024)].copy()

# ===========================================================================
# 304.2 EVOLUCIÓN DE LA SINIESTRALIDAD (VOLUMEN Y TENDENCIA)
# ===========================================================================
plt.figure(figsize=(15, 6))
df_ts.groupby('Year').size().plot(kind='line', marker='o', color='midnightblue', linewidth=2)
plt.title('Evolución Anual de la Siniestralidad: Identificación de Quiebres Estructurales', fontsize=14)
plt.ylabel('Cantidad de Siniestros')
plt.grid(True, alpha=0.3)
plt.show()

# ===========================================================================
# 304.3 RATIO DE ÉXITO (ANCR) A LO LARGO DEL TIEMPO
# ===========================================================================
# El ratio de establecimiento legal es el proxy de eficiencia del sistema
df_ts['Has_ANCR'] = df_ts['Interval Assembled to ANCR'].notnull().astype(int)
ancr_trend = df_ts.groupby('Year')['Has_ANCR'].mean() * 100

plt.figure(figsize=(15, 6))
ancr_trend.plot(kind='bar', color='teal', alpha=0.7)
plt.title('Evolución del Ratio de Establecimiento Legal (ANCR %)', fontsize=14)
plt.ylabel('Porcentaje de Casos Establecidos (%)')
plt.axhline(ancr_trend.mean(), color='red', linestyle='--', label=f'Promedio: {ancr_trend.mean():.1f}%')
plt.legend()
plt.show()